# Hexapawn: IA vs. IA

Duas IAs (vermelha e azul) treinam jogando uma contra a outra. Escolha o número de partidas e clique em ▶ para acompanhar o aprendizado.

> Este notebook usa widgets interativos do Google Colab. Para jogar, abra no Colab e execute todas as células (Ambiente de execução → Executar tudo).

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import random
import time
from google.colab import output


vitorias_vermelho  = 0
vitorias_azul      = 0
log_vitorias       = []
log_movimentos     = []
contador_jogadas   = 0

caixinhas_vermelho = {}
caixinhas_azul     = {}
historico_vermelho = []
historico_azul     = []

tab_inicial = [['W','W','W'], [' ',' ',' '], ['B','B','B']]
tabuleiro   = [linha[:] for linha in tab_inicial]
tela_saida  = widgets.Output()
rodando     = False

In [ ]:
############ mov ##############
def buscar_movimentos(tab, cor):
    lista   = []
    direcao = 1  if cor == 'W' else -1
    inimigo = 'B' if cor == 'W' else 'W'
    for r in range(3):
        for c in range(3):
            if tab[r][c] == cor:
                nr = r + direcao
                if 0 <= nr < 3:
                    if tab[nr][c] == ' ':
                        lista.append(((r,c),(nr,c)))
                    for dc in [-1,1]:
                        nc = c + dc
                        if 0 <= nc < 3 and tab[nr][nc] == inimigo:
                            lista.append(((r,c),(nr,nc)))
    return lista

In [ ]:
########## vermelho #######################
def vermelho_escolhe_jogada(tab):
    global contador_jogadas
    estado = str([linha[:] for linha in tab])
    if estado not in caixinhas_vermelho:
        caixinhas_vermelho[estado] = buscar_movimentos(tab, 'W')
    opcoes = caixinhas_vermelho[estado]
    if not opcoes:
        return None
    escolha = random.choice(opcoes)
    historico_vermelho.append((estado, escolha))
    contador_jogadas += 1
    return escolha

def atualizar_aprendizado_vermelho(ganhou):
    global vitorias_vermelho, vitorias_azul
    if historico_vermelho:
        if ganhou:
            for estado, mov in historico_vermelho:
                if estado in caixinhas_vermelho:
                    caixinhas_vermelho[estado].append(mov)
        else:
            estado, mov = historico_vermelho[-1]
            if estado in caixinhas_vermelho and mov in caixinhas_vermelho[estado]:
                caixinhas_vermelho[estado].remove(mov)

In [ ]:
########## azul #######################
def azul_escolhe_jogada(tab):
    global contador_jogadas
    estado = str([linha[:] for linha in tab])
    if estado not in caixinhas_azul:
        caixinhas_azul[estado] = buscar_movimentos(tab, 'B')
    opcoes = caixinhas_azul[estado]
    if not opcoes:
        return None
    escolha = random.choice(opcoes)
    historico_azul.append((estado, escolha))
    contador_jogadas += 1
    return escolha

def atualizar_aprendizado_azul(ganhou):
    global vitorias_azul
    if historico_azul:
        if ganhou:
            for estado, mov in historico_azul:
                if estado in caixinhas_azul:
                    caixinhas_azul[estado].append(mov)
        else:
            estado, mov = historico_azul[-1]
            if estado in caixinhas_azul and mov in caixinhas_azul[estado]:
                caixinhas_azul[estado].remove(mov)

In [ ]:
######### caixinhas ###########
def checar_fim(tab):
    if any(tab[2][c] == 'W' for c in range(3)):
        return 'V'
    if any(tab[0][c] == 'B' for c in range(3)):
        return 'A'
    if not buscar_movimentos(tab, 'W'):
        return 'A'
    if not buscar_movimentos(tab, 'B'):
        return 'V'
    return None

def registrar(vencedor):
    global vitorias_vermelho, vitorias_azul, contador_jogadas
    atualizar_aprendizado_vermelho(vencedor == 'V')
    atualizar_aprendizado_azul(vencedor == 'A')
    if vencedor == 'V': vitorias_vermelho += 1
    else:               vitorias_azul     += 1
    log_vitorias.append(vencedor)
    log_movimentos.append(contador_jogadas)
    historico_vermelho.clear()
    historico_azul.clear()

In [ ]:
########### visual #################
def desenhar_tabuleiro(msg=""):
    with tela_saida:
        clear_output(wait=True)
        partida_atual = len(log_vitorias) + 1
        print(f"PARTIDA {partida_atual}  —  {msg}")
        grade = widgets.GridspecLayout(3, 3, width='300px', height='300px')
        for r in range(3):
            for c in range(3):
                p   = tabuleiro[r][c]
                cor = '#EEEEEE' if (r+c) % 2 == 0 else '#CCCCCC'
                if p == 'W': cor = '#FF3333'
                elif p == 'B': cor = '#0055FF'
                b = widgets.Button(
                    description=p,
                    layout=widgets.Layout(height='95px', width='95px')
                )
                b.style.button_color = cor
                b.style.text_color   = 'white'
                b.style.font_weight  = 'bold'
                grade[r, c] = b
        display(grade)
        mostrar_graficos()

In [ ]:
############ graf ############
def mostrar_graficos():
    if not log_vitorias: return
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3))
    fig.patch.set_facecolor('white')

    cores_linha = ['#FF3333' if v == 'V' else '#0055FF' for v in log_vitorias]
    ys          = [1 if v == 'V' else 0 for v in log_vitorias]
    xs          = list(range(1, len(ys)+1))
    for i in range(len(xs)-1):
        ax1.plot(xs[i:i+2], ys[i:i+2], color=cores_linha[i], linewidth=2)
    ax1.scatter(xs, ys, c=cores_linha, s=30, zorder=5)
    ax1.set_yticks([0,1])
    ax1.set_yticklabels(['Azul','Vermelho'], fontsize=8)
    ax1.set_title("Histórico de vitórias por partida", fontsize=9)
    ax1.set_xlabel("Partida", fontsize=8)
    ax1.set_facecolor('white')
    ax1.grid(True, linestyle=':', alpha=0.5)
    v_patch = mpatches.Patch(color='#FF3333', label=f'Vermelho {vitorias_vermelho}')
    a_patch = mpatches.Patch(color='#0055FF', label=f'Azul {vitorias_azul}')
    ax1.legend(handles=[v_patch, a_patch], fontsize=8)

    ax2.bar(['Vermelho','Azul'], [vitorias_vermelho, vitorias_azul],
            color=['#FF3333','#0055FF'], edgecolor='white')
    ax2.set_title(f"Placar geral  {vitorias_vermelho} × {vitorias_azul}", fontsize=9)
    ax2.set_facecolor('white')
    ax2.yaxis.get_major_locator().set_params(integer=True)

    plt.tight_layout()
    plt.show()

In [ ]:
############ jogo ############
def jogar_partida(animar):
    global tabuleiro, contador_jogadas
    tabuleiro          = [linha[:] for linha in tab_inicial]
    contador_jogadas   = 0
    vez                = 'W'

    for _ in range(50):
        fim = checar_fim(tabuleiro)
        if fim:
            return fim

        if vez == 'W':
            mov = vermelho_escolhe_jogada(tabuleiro)
            if mov is None: return 'A'
            (r1,c1),(r2,c2) = mov
            tabuleiro[r1][c1] = ' '
            tabuleiro[r2][c2] = 'W'
            vez = 'B'
        else:
            mov = azul_escolhe_jogada(tabuleiro)
            if mov is None: return 'V'
            (r1,c1),(r2,c2) = mov
            tabuleiro[r1][c1] = ' '
            tabuleiro[r2][c2] = 'B'
            vez = 'W'

        if animar:
            desenhar_tabuleiro(f"{'Vermelho' if vez=='B' else 'Azul'} moveu")
            time.sleep(0.35)

    return checar_fim(tabuleiro) or 'A'

def iniciar_treino(b):
    global rodando
    if rodando: return
    rodando = True
    btn_treinar.disabled = True

    try:
        n = int(campo_partidas.value)
    except ValueError:
        n = 100

    animar = chk_animar.value

    for i in range(n):
        vencedor = jogar_partida(animar)
        registrar(vencedor)
        if not animar and (i+1) % max(1, n//20) == 0:
            desenhar_tabuleiro(f"Treinando… {i+1}/{n}")

    tabuleiro[:] = [linha[:] for linha in tab_inicial]
    desenhar_tabuleiro(f"Treino concluído! {n} partidas")
    rodando = False
    btn_treinar.disabled = False

###############################################


campo_partidas = widgets.IntText(
    value=100,
    description='Partidas:',
    layout=widgets.Layout(width='160px')
)
chk_animar = widgets.Checkbox(
    value=True,
    description='Animar tabuleiro',
    indent=False
)
btn_treinar = widgets.Button(
    description='▶',
    button_style='danger',
    layout=widgets.Layout(width='130px', height='36px')
)
btn_treinar.on_click(iniciar_treino)

controles = widgets.HBox(
    [campo_partidas, chk_animar, btn_treinar],
    layout=widgets.Layout(margin='0 0 10px 0', gap='12px')
)

display(controles, tela_saida)
desenhar_tabuleiro("Pronto para treinar!")